In [1]:
import pandas as pd
import numpy as np
import boto3

bucket = "insurance-pricing-model-akshayaa"

# Load raw data
freq = pd.read_csv(f"s3://{bucket}/freMTPL2freq.csv")
sev = pd.read_csv(f"s3://{bucket}/freMTPL2sev.csv")

C:\Users\prish\OneDrive\Desktop\Insurance Project\Insurance-Underwriting-Pricing-Model\venv\lib\site-packages\fsspec\registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [2]:
df_freq=pd.DataFrame(freq)
df_sev=pd.DataFrame(sev)

In [3]:
df_freq.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,Regular,1217,R82
1,3.0,1,0.77,D,5,0,55,50,B12,Regular,1217,R82
2,5.0,1,0.75,B,6,2,52,50,B12,Diesel,54,R22
3,10.0,1,0.09,B,7,0,46,50,B12,Diesel,76,R72
4,11.0,1,0.84,B,7,0,46,50,B12,Diesel,76,R72


### Cleaning

In [4]:
# Type consistency
df_freq['IDpol'] = df_freq['IDpol'].astype(int)
df_sev['IDpol'] = df_sev['IDpol'].astype(int)

In [5]:
# Dropping Area to avoid multicollinearity with Density
df_freq = df_freq.drop(columns=['Area'])

# Log transforming Desnity to log(Density)
df_freq['log_density'] = np.log1p(df_freq['Density'])

# Dropping density after transformation
df_freq = df_freq.drop(columns=['Density'])

In [6]:
# Capping outliers to 5 for ClaimNB
df_freq['ClaimNb'] = df_freq['ClaimNb'].clip(upper=5)

# Filtering out rows for which Exposure <= 1.0
df_freq=df_freq[df_freq['Exposure']<=1.0]
print(df_freq.shape)

(676789, 11)


### Joining two datasets on IDpol

In [7]:
sev_agg = sev.groupby('IDpol', as_index=False)['ClaimAmount'].sum()
df_joined = df_freq.merge(sev_agg, on='IDpol', how='left')
df_joined['ClaimAmount'] = df_joined['ClaimAmount'].fillna(0)

print("df_joined shape:", df_joined.shape)
print(df_joined.head())

df_joined shape: (676789, 12)
   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
0      1        1      0.10         5       0       55          50      B12   
1      3        1      0.77         5       0       55          50      B12   
2      5        1      0.75         6       2       52          50      B12   
3     10        1      0.09         7       0       46          50      B12   
4     11        1      0.84         7       0       46          50      B12   

    VehGas Region  log_density  ClaimAmount  
0  Regular    R82     7.104965          0.0  
1  Regular    R82     7.104965          0.0  
2   Diesel    R22     4.007333          0.0  
3   Diesel    R72     4.343805          0.0  
4   Diesel    R72     4.343805          0.0  


### Dealing with Vehicle Brands: grouping small/rare categories together "Other"; threshold is <2-3% of the portfolio. 

In [8]:
print(df_joined['VehBrand'].value_counts())

VehBrand
B12    165959
B1     162407
B2     159491
B3      53266
B5      34692
B6      28483
B4      25143
B10     17640
B11     13531
B13     12147
B14      4030
Name: count, dtype: int64


In [9]:
threshold = 0.02  # group any brand representing less than 2% of total policies
brand_counts = df_joined['VehBrand'].value_counts(normalize=True)
rare_brands = brand_counts[brand_counts < threshold].index.tolist()

print("Brands to be grouped as 'Other':", rare_brands)
print(f"Combined 'Other' share: {brand_counts[rare_brands].sum()*100:.2f}%")

Brands to be grouped as 'Other': ['B11', 'B13', 'B14']
Combined 'Other' share: 4.39%


In [10]:
df_joined['VehBrand_grouped'] = df_joined['VehBrand'].apply(lambda x: 'Other' if x in rare_brands else x)

print(df_joined['VehBrand_grouped'].value_counts())

VehBrand_grouped
B12      165959
B1       162407
B2       159491
B3        53266
B5        34692
Other     29708
B6        28483
B4        25143
B10       17640
Name: count, dtype: int64


In [11]:
df_joined['VehGas_binary'] = (df_joined['VehGas'] == 'Diesel').astype(int)
# VehGas_binary = 1 represents Diesel

print(df_joined[['VehGas', 'VehGas_binary']].drop_duplicates())

    VehGas  VehGas_binary
0  Regular              0
2   Diesel              1


In [12]:
df_encoded = pd.get_dummies(df_joined, columns=['VehBrand_grouped', 'Region'], drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

(676789, 41)
['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'log_density', 'ClaimAmount', 'VehGas_binary', 'VehBrand_grouped_B10', 'VehBrand_grouped_B12', 'VehBrand_grouped_B2', 'VehBrand_grouped_B3', 'VehBrand_grouped_B4', 'VehBrand_grouped_B5', 'VehBrand_grouped_B6', 'VehBrand_grouped_Other', 'Region_R21', 'Region_R22', 'Region_R23', 'Region_R24', 'Region_R25', 'Region_R26', 'Region_R31', 'Region_R41', 'Region_R42', 'Region_R43', 'Region_R52', 'Region_R53', 'Region_R54', 'Region_R72', 'Region_R73', 'Region_R74', 'Region_R82', 'Region_R83', 'Region_R91', 'Region_R93', 'Region_R94']


In [13]:
df_encoded.head()

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,log_density,...,Region_R53,Region_R54,Region_R72,Region_R73,Region_R74,Region_R82,Region_R83,Region_R91,Region_R93,Region_R94
0,1,1,0.10,5,0,55,50,B12,Regular,7.104965,...,False,False,False,False,False,True,False,False,False,False
1,3,1,0.77,5,0,55,50,B12,Regular,7.104965,...,False,False,False,False,False,True,False,False,False,False
2,5,1,0.75,6,2,52,50,B12,Diesel,4.007333,...,False,False,False,False,False,False,False,False,False,False
3,10,1,0.09,7,0,46,50,B12,Diesel,4.343805,...,False,False,True,False,False,False,False,False,False,False
4,11,1,0.84,7,0,46,50,B12,Diesel,4.343805,...,False,False,True,False,False,False,False,False,False,False


In [14]:
dummy_cols = [col for col in df_encoded.columns if col.startswith('VehBrand_grouped_') or col.startswith('Region_')]
df_encoded[dummy_cols] = df_encoded[dummy_cols].astype(int)

print(df_encoded[dummy_cols].head())

   VehBrand_grouped_B10  VehBrand_grouped_B12  VehBrand_grouped_B2  \
0                     0                     1                    0   
1                     0                     1                    0   
2                     0                     1                    0   
3                     0                     1                    0   
4                     0                     1                    0   

   VehBrand_grouped_B3  VehBrand_grouped_B4  VehBrand_grouped_B5  \
0                    0                    0                    0   
1                    0                    0                    0   
2                    0                    0                    0   
3                    0                    0                    0   
4                    0                    0                    0   

   VehBrand_grouped_B6  VehBrand_grouped_Other  Region_R21  Region_R22  ...  \
0                    0                       0           0           0  ...   
1           

In [15]:
df_encoded = df_encoded.drop(columns=['VehBrand', 'VehGas'])
print(df_encoded.shape)
print(df_encoded.columns.tolist())

(676789, 39)
['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density', 'ClaimAmount', 'VehGas_binary', 'VehBrand_grouped_B10', 'VehBrand_grouped_B12', 'VehBrand_grouped_B2', 'VehBrand_grouped_B3', 'VehBrand_grouped_B4', 'VehBrand_grouped_B5', 'VehBrand_grouped_B6', 'VehBrand_grouped_Other', 'Region_R21', 'Region_R22', 'Region_R23', 'Region_R24', 'Region_R25', 'Region_R26', 'Region_R31', 'Region_R41', 'Region_R42', 'Region_R43', 'Region_R52', 'Region_R53', 'Region_R54', 'Region_R72', 'Region_R73', 'Region_R74', 'Region_R82', 'Region_R83', 'Region_R91', 'Region_R93', 'Region_R94']


In [16]:
pd.set_option('display.max_columns', None)
print(df_encoded.head())

   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus  \
0      1        1      0.10         5       0       55          50   
1      3        1      0.77         5       0       55          50   
2      5        1      0.75         6       2       52          50   
3     10        1      0.09         7       0       46          50   
4     11        1      0.84         7       0       46          50   

   log_density  ClaimAmount  VehGas_binary  VehBrand_grouped_B10  \
0     7.104965          0.0              0                     0   
1     7.104965          0.0              0                     0   
2     4.007333          0.0              1                     0   
3     4.343805          0.0              1                     0   
4     4.343805          0.0              1                     0   

   VehBrand_grouped_B12  VehBrand_grouped_B2  VehBrand_grouped_B3  \
0                     1                    0                    0   
1                     1         

### Frequency & Severity Modeling with train test split

In [18]:
freq_data = df_encoded.copy()
severity_data = df_encoded[(df_encoded['ClaimNb'] > 0) & (df_encoded['ClaimAmount'] > 0)]

print("Frequency population:", freq_data.shape)
print("Severity population:", severity_data.shape)

Frequency population: (676789, 39)
Severity population: (24893, 39)


In [20]:
from sklearn.model_selection import train_test_split

freq_train, freq_test = train_test_split(freq_data, test_size=0.2, random_state=42)
sev_train, sev_test = train_test_split(severity_data, test_size=0.2, random_state=42)

print("Freq train/test: ", freq_train.shape, freq_test.shape)
print("Severity train/test: ", sev_train.shape, sev_test.shape)

Freq train/test:  (541431, 39) (135358, 39)
Severity train/test:  (19914, 39) (4979, 39)


### Frequency Modeling

#### Poisson GLM - baseline model

In [21]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Define feature columns (exclude IDpol, targets, and raw Exposure since it's used as offset)
exclude_cols = ['IDpol', 'ClaimNb', 'ClaimAmount', 'Exposure']
feature_cols = [col for col in freq_train.columns if col not in exclude_cols]

X_train = freq_train[feature_cols]
y_train = freq_train['ClaimNb']
offset_train = np.log(freq_train['Exposure'])

X_train_const = sm.add_constant(X_train)

poisson_model = sm.GLM(y_train, X_train_const, family=sm.families.Poisson(), offset=offset_train)
poisson_results = poisson_model.fit()

print(poisson_results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               541431
Model:                            GLM   Df Residuals:                   541395
Model Family:                 Poisson   Df Model:                           35
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1464e+05
Date:                Sat, 19 Sep 2026   Deviance:                   1.7369e+05
Time:                        00:47:18   Pearson chi2:                 1.36e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01050
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -4

### Conclusion: Which features actually matter?  
VehiclePower, VehicleAge, Driver’s Age, BonusMalus, log_Density, VehicleGas. <br>
Vehicle Brand: B12 and B5 <br>
Region-R24, Region-R41, Region-R53, Region-R82, Region-R83. <br>
Note: Log-transforming skewness rarely moves accuracy much in a GLM, but it's essential for correct coefficient magnitude, correct significance testing. The log version just does it more efficiently (fewer extreme residuals), so predictions at very low and very high densities are better calibrated.

### Inference:
Keeping all other features constant, 1-unit increase in BonusMalus multiplies expected claim frequency by exp(0.0223) ≈ 1.0226. <br>
Since -0.0445 is negative, switching from Regular (0) to Diesel (1) decreases the log of expected claim frequency. I.e. Diesel vehicles have lower claim frequency. <br>
Pseudo R² value - 0.0105 is a roughly weak signal (between 0.20 and 0.40 is considered good). <br>

### Actionable Business Insights from GLM modeling:
Pricing: The insurer should seriously consider higher premiums for policyholders with the B12 vehicle brand and those living in regions like R82, R24 and R53. <br>
BonusMalus feature is a very strong predictor; a high BonusMalus score reliably signals higher claim frequency. So, insurers need to watch out for policies that have a high BonusMalus score. <br> 
Underwriting should tighten appetite in these high-frequency cells and grow selectively by offering discounts in low-risk regions like R41 and R83. <br>
